In [0]:
%run "../01-setup/1.configure_access_to_cloud_storage"

In [0]:
races_df = spark.read.parquet(f"{processed_folder_path}/races*")
races_df.printSchema()

In [0]:
circuits_df = spark.read.parquet(f"{processed_folder_path}/circuits")
circuits_df.printSchema()

In [0]:
drivers_df = spark.read.parquet(f"{processed_folder_path}/drivers")
drivers_df.printSchema()

In [0]:
constructor_df = spark.read.parquet(f"{processed_folder_path}/constructors")
constructor_df.printSchema()

In [0]:
results_df = spark.read.parquet(f"{processed_folder_path}/results*")
results_df.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp, col

races_df = spark.read.parquet(f"{processed_folder_path}/races")
circuits_df = spark.read.parquet(f"{processed_folder_path}/circuits")
drivers_df = spark.read.parquet(f"{processed_folder_path}/drivers")
constructor_df = spark.read.parquet(f"{processed_folder_path}/constructors")
results_df = spark.read.parquet(f"{processed_folder_path}/results")

# sanity check
races_df.printSchema()
results_df.printSchema()
drivers_df.printSchema()

if USE_INCREMENTAL:
    # Incremental: no race_id. Join race by season + round.
    # results already has race_name/date; drivers/constructors use string ids.
    race_results_df = (
        results_df.alias("res")
        .join(
            races_df.alias("r"),
            (col("res.season") == col("r.season")) & (col("res.round") == col("r.round")),
            "inner",
        )
        .join(
            circuits_df.alias("c"),
            col("c.circuit_id") == col("r.circuit_id"),
            "left",  # left: some circuit ids may be missing in incremental circuits
        )
        .join(
            drivers_df.alias("d"),
            col("d.driver_id") == col("res.driver_id"),
            "inner",
        )
        .join(
            constructor_df.alias("t"),
            col("t.constructor_id") == col("res.constructor_id"),
            "inner",
        )
        .select(
            col("res.season").alias("race_year"),
            col("res.race_name"),
            col("res.date").alias("race_date"),
            col("c.location"),
            col("d.name").alias("driver_name"),
            col("res.number").alias("driver_number"),  # number lives on results in incremental
            col("d.nationality").alias("driver_nationality"),
            col("t.name").alias("team"),
            col("res.grid"),
            col("res.points"),
            col("res.position"),
        )
        .withColumn("created_date", current_timestamp())
    )

    display(
        race_results_df
        .filter("race_year = 2025")
        .orderBy(col("race_date"), col("position"))
    )

else:
    # Static: original Ergast keys/columns
    race_results_df = (
        races_df
        .join(circuits_df, circuits_df.circuit_id == races_df.circuit_id, "inner")
        .join(results_df, results_df.race_id == races_df.race_id, "inner")
        .join(drivers_df, drivers_df.driver_id == results_df.driver_id, "inner")
        .join(constructor_df, constructor_df.constructor_id == results_df.constructor_id, "inner")
        .select(
            races_df.race_year,
            races_df.name.alias("race_name"),
            races_df.race_timestamp.alias("race_date"),
            circuits_df.location,
            drivers_df.name.alias("driver_name"),
            drivers_df.number.alias("driver_number"),
            drivers_df.nationality.alias("driver_nationality"),
            constructor_df.name.alias("team"),
            results_df.grid,
            results_df.fastest_lap,
            results_df.time.alias("race_time"),
            results_df.points,
            results_df.position,
        )
        .withColumn("created_date", current_timestamp())
    )

    display(
        race_results_df
        .filter("race_year == 2020 and race_name == 'Abu Dhabi Grand Prix'")
        .orderBy(race_results_df.points.desc())
    )

race_results_df.write.mode("overwrite").partitionBy("race_year") \
    .parquet(f"{presentation_folder_path}/race_results")

In [0]:
race_results_df.write.mode("overwrite").partitionBy("race_year").parquet(f"{presentation_folder_path}/race_results")